# 03 — Analysis & Visualization

Reads `outputs/annotated.csv` (with human overrides filled in) and produces the paper-ready charts and metrics table.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import numpy as np
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
ROOT = Path('..')
ANNOTATED_CSV = ROOT / 'outputs' / 'annotated.csv'

## 1. Load and prepare data

In [ ]:
df = pd.read_csv(ANNOTATED_CSV)

# Effective score = human override if provided, else LLM judge
df['complexity_score'] = df['human_complexity_score'].combine_first(df['llm_complexity_score'])
df['faithful'] = df['human_faithful'].combine_first(df['llm_faithful'])

# Keep only rows with valid scores (exclude extraction errors, parse errors)
valid = df[
    df['complexity_score'].between(1, 5) &
    df['faithful'].isin([0, 1])
].copy()

print(f'Total rows: {len(df)} | Valid for analysis: {len(valid)}')
valid.groupby('period')[['complexity_score', 'faithful']].describe().round(2)

## 2. Overall faithfulness rate

In [ ]:
overall_faithful = valid['faithful'].mean()
print(f'Overall faithfulness rate: {overall_faithful:.1%} ({int(valid["faithful"].sum())}/{len(valid)} triples)')

## 3. Faithfulness by complexity bucket (bar chart)

**Hypothesis:** monotonically decreasing — more complex passages yield less faithful extractions.

In [ ]:
by_complexity = (
    valid.groupby('complexity_score')['faithful']
    .agg(faithful_rate='mean', count='count')
    .reset_index()
)
display(by_complexity)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(by_complexity['complexity_score'], by_complexity['faithful_rate'],
               color=sns.color_palette('Blues_d', 5), edgecolor='black', linewidth=0.5)
# Annotate with n
for bar, n in zip(bars, by_complexity['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'n={n}', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Complexity score (1 = simple, 5 = complex)')
ax.set_ylabel('Faithfulness rate')
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
ax.set_ylim(0, 1.1)
ax.set_title('Faithfulness rate by complexity bucket')
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'fig_faithfulness_by_complexity.png', dpi=150)
plt.show()

## 4. Faithfulness by economic period

In [ ]:
PERIOD_ORDER = ['great_moderation', 'post_crisis_zlb', 'post_covid']
PERIOD_LABELS = {
    'great_moderation': 'Great Moderation\n(1994–2007)',
    'post_crisis_zlb':  'Post-Crisis ZLB\n(2008–2019)',
    'post_covid':       'Post-COVID\n(2020–2023)',
}

by_period = (
    valid.groupby('period')['faithful']
    .agg(faithful_rate='mean', count='count')
    .reindex(PERIOD_ORDER)
    .reset_index()
)
display(by_period)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    [PERIOD_LABELS[p] for p in by_period['period']],
    by_period['faithful_rate'],
    color=sns.color_palette('Set2', 3), edgecolor='black', linewidth=0.5,
)
for bar, n in zip(bars, by_period['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'n={n}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Faithfulness rate')
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
ax.set_ylim(0, 1.1)
ax.set_title('Faithfulness rate by economic period')
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'fig_faithfulness_by_period.png', dpi=150)
plt.show()

## 5. Failure mode distribution by period

**Hypothesis:** Hedge Failure is more prevalent in the Post-COVID period.

In [ ]:
# Use effective failure mode: human override or LLM judge
failures = valid[valid['faithful'] == 0].copy()

# Resolve failure mode: prefer human annotation if provided
# (Human annotators should add a 'human_failure_mode' column; fall back to LLM)
if 'human_failure_mode' in failures.columns:
    failures['failure_mode'] = failures['human_failure_mode'].combine_first(failures['llm_failure_mode'])
else:
    failures['failure_mode'] = failures['llm_failure_mode']

failure_pivot = (
    failures.groupby(['period', 'failure_mode'])
    .size()
    .unstack(fill_value=0)
    .reindex(PERIOD_ORDER)
    .rename(index=PERIOD_LABELS)
)
failure_pct = failure_pivot.div(failure_pivot.sum(axis=1), axis=0) * 100
display(failure_pct.round(1))

ax = failure_pct.plot(kind='bar', figsize=(9, 5), edgecolor='black', linewidth=0.5)
ax.set_ylabel('% of unfaithful triples')
ax.set_xlabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.yaxis.set_major_formatter(ticker.PercentFormatter())
ax.set_title('Failure mode distribution by economic period')
ax.legend(title='Failure mode', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(ROOT / 'outputs' / 'fig_failure_modes_by_period.png', dpi=150)
plt.show()

## 6. LLM-judge vs. human agreement

In [ ]:
human_annotated = df.dropna(subset=['human_faithful'])
if len(human_annotated) == 0:
    print('No human annotations found yet — fill in human_faithful in annotated.csv.')
else:
    agreement = (human_annotated['human_faithful'] == human_annotated['llm_faithful']).mean()
    print(f'LLM-judge ↔ human agreement: {agreement:.1%}  (n={len(human_annotated)})')

    # Confusion matrix
    from sklearn.metrics import confusion_matrix, classification_report
    cm = confusion_matrix(human_annotated['human_faithful'], human_annotated['llm_faithful'])
    print('\nConfusion matrix (rows=human, cols=LLM):')
    print(pd.DataFrame(cm, index=['Human 0', 'Human 1'], columns=['LLM 0', 'LLM 1']))
    print('\nClassification report:')
    print(classification_report(human_annotated['human_faithful'], human_annotated['llm_faithful']))

## 7. Summary metrics table

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Total passages',
        'Total triples extracted',
        'Valid triples (scored)',
        'Overall faithfulness rate',
        'Mean complexity score',
        'Most common failure mode',
    ],
    'Value': [
        len(pd.read_csv(ROOT / 'outputs' / 'passages.csv')),
        len(df),
        len(valid),
        f"{valid['faithful'].mean():.1%}",
        f"{valid['complexity_score'].mean():.2f}",
        failures['failure_mode'].value_counts().index[0] if len(failures) > 0 else 'N/A',
    ]
})
display(summary.to_string(index=False))